#### Import libraries

In [ ]:
import json
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

In [ ]:
paths = {
  "code": '/home/zanardi/Codes/TC/HyperNet/hypernet',                           # Path to folder containing HyperNet
  "cfg": "/home/zanardi/Codes/TC/HyperNet/hypernet/exec/configfile.json"        # Path to HyperNet configuration file
}
import os
import sys
sys.path.append(paths["code"])
os.environ.update({"HYPNET_CONFIGFILE": paths["cfg"]})

In [ ]:
from hypernet.apps import utils as autils
from hypernet import backend as bkd
from hypernet import const

#### Thermophysics

In [ ]:
with open("./inputs.json") as file:
    inputs = json.load(file)

In [ ]:
# Species thermo
sp_th = autils._get_specie_thermos(inputs)
nb_bins = sp_th["O2"].specie.nb_comp

In [ ]:
# Mixture
mix = autils._get_mixture(inputs, sp_th)

In [ ]:
# Kinetics
kin = autils._get_kinetics(inputs, sp_th, mix)

#### Solution

In [ ]:
sol = pd.read_csv("./ref_sol.dat", skiprows=2, header=None, delimiter="  ", engine='python')
mass_cols = ["X_O"] + [f"X_O{i+1}" for i in range(nb_bins)]
sol.columns = ["t"] + mass_cols + ["T", "rho", "p", "n", "E"]

#### Check source term

In [ ]:
n = sol[mass_cols].to_numpy() * sol[["n"]].to_numpy()
T = sol['T'].to_numpy().reshape(-1,1)
t = sol['t'].to_numpy().reshape(-1,1)

In [ ]:
dt = np.roll(t,-1,axis=0) - t
dn = np.roll(n,-1,axis=0) - n
wt = dn / dt / const.UNA

In [ ]:
w = kin.function(
    t=bkd.to_backend(t),
    y=[bkd.to_backend(n),bkd.to_backend(T)]
)
wp = bkd.to_numpy(w/const.UNA)[:,:-1]

In [ ]:
def plot_source(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$\omega$'],
    scales=['log', 'linear'],
    names=None
):
    # Initialize figures
    fig, ax = plt.subplots()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    ax.plot(x, y, lw=1.0, label=names)
    if (xref is not None):
        ax.plot(xref, yref, lw=1.0, ls='--')
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    plt.show()
    plt.close()

In [ ]:
plot_source(
    t.squeeze()[2:-2],
    wp[2:-2],
    xref=t.squeeze()[2:-2],
    yref=wt[2:-2],
    names=['O'] + [r'O$_2^{(%d)}$' % (i+1) for i in range(nb_bins)]
)